# 🔍 Process Mining Workshop — PM4Py, Point & Click

**You will not write or edit any Python code in this notebook.** Every piece
of analysis happens by clicking buttons. The code cells exist so the
computer can run things — you can collapse or ignore them.

### How to use this notebook
1. Click **Runtime → Run all** (Google Colab) or **Cell → Run All** (Jupyter) once, top to bottom.
   The first cell installs a few packages and may take 1–2 minutes — that's normal.
2. Scroll down to **Step 1** and either upload your own event log (a CSV export) or click
   **"Use sample data instead"** to try it immediately with a small built-in example.
3. Scroll to **Step 2** and click the buttons to explore the process — a map of how work
   actually flows, where the slow points are, how many different paths people take, and so on.
4. There's a plain-language glossary near the bottom if any term is unfamiliar.

> 💡 **Tip:** the easiest way to run this with zero setup is to open it in
> **[Google Colab](https://colab.research.google.com/)** (File → Upload notebook, or drag
> this `.ipynb` file onto the Colab home page). Colab runs entirely in your browser —
> nothing to install on your own laptop.

---


## ⚙️ Setup (run once)

This installs the process-mining library (**PM4Py**) and the interactive
button/widget library used throughout this notebook. You only need to run
this cell once per session — after that, just re-run the cells below it as
many times as you like.

In [1]:
# Run this cell first. It may take 1-2 minutes the first time — that's normal.
import sys, subprocess

def _pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _pkg in ["pm4py", "ipywidgets", "pandas", "graphviz"]:
    _pip_install(_pkg)

# Graphviz's drawing engine (the "dot" program) needs to exist on the system,
# not just the Python wrapper. This installs it on Debian/Ubuntu-based
# notebook environments (e.g. Google Colab); it's a no-op elsewhere and is
# safe to ignore if it fails (e.g. on Windows or if it's already installed).
try:
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False,
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception:
    pass

print("✅ Setup complete — you can now run the cells below.")

✅ Setup complete — you can now run the cells below.


In [2]:
# Imports used throughout the notebook.
import io
import re
import warnings
from datetime import datetime

import pandas as pd
import numpy as np
import pm4py
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown, Image

warnings.filterwarnings("ignore")
pd.set_option("mode.chained_assignment", None)

# All state for the workshop lives in this one dictionary so every button's
# click-handler can read/write it without needing global variables everywhere.
state = {"df": None, "case_col": None, "activity_col": None, "time_col": None,
         "resource_col": None, "source_name": None}

print("Ready.")



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Ready.


---
## 📥 Step 1 — Load an Event Log

A process-mining "event log" is just a table where **each row is one step
that happened**, with at minimum:

| Case ID | Activity | Timestamp |
|---|---|---|
| Which case/order/ticket/patient this step belongs to | What happened | When it happened |

Upload a CSV export of something like this (an order process, a support
ticket queue, an approval workflow, a patient journey — anything with
repeated cases going through steps over time), or click **"Use sample data
instead"** to try everything out immediately.

In [3]:
# ---- Sample dataset (small, built-in order-to-cash process) -------------
# This is embedded directly in the notebook so "Use sample data" always
# works, even with no internet access or uploaded file.
_SAMPLE_CSV = """case_id,activity,timestamp,resource,department
ORD-001,Register Order,2024-01-02 09:00,Alice,Sales
ORD-001,Check Stock,2024-01-02 10:15,Bob,Warehouse
ORD-001,Approve Payment,2024-01-02 14:30,Carol,Finance
ORD-001,Ship Order,2024-01-03 08:00,Dave,Warehouse
ORD-001,Close Order,2024-01-05 09:00,Alice,Sales
ORD-002,Register Order,2024-01-02 09:20,Alice,Sales
ORD-002,Check Stock,2024-01-02 11:00,Bob,Warehouse
ORD-002,Backorder Item,2024-01-04 09:00,Bob,Warehouse
ORD-002,Check Stock,2024-01-06 09:00,Bob,Warehouse
ORD-002,Approve Payment,2024-01-06 15:00,Carol,Finance
ORD-002,Ship Order,2024-01-07 08:30,Dave,Warehouse
ORD-002,Close Order,2024-01-09 09:00,Alice,Sales
ORD-003,Register Order,2024-01-03 09:10,Eve,Sales
ORD-003,Check Stock,2024-01-03 09:45,Bob,Warehouse
ORD-003,Approve Payment,2024-01-03 10:30,Carol,Finance
ORD-003,Ship Order,2024-01-03 16:00,Dave,Warehouse
ORD-003,Close Order,2024-01-04 09:00,Eve,Sales
ORD-004,Register Order,2024-01-04 09:00,Alice,Sales
ORD-004,Check Stock,2024-01-04 09:40,Bob,Warehouse
ORD-004,Approve Payment,2024-01-04 10:00,Carol,Finance
ORD-004,Payment Rejected,2024-01-04 13:00,Carol,Finance
ORD-004,Approve Payment,2024-01-05 09:00,Carol,Finance
ORD-004,Ship Order,2024-01-05 15:00,Dave,Warehouse
ORD-004,Close Order,2024-01-06 09:00,Alice,Sales
ORD-005,Register Order,2024-01-05 09:05,Eve,Sales
ORD-005,Check Stock,2024-01-05 09:30,Bob,Warehouse
ORD-005,Approve Payment,2024-01-05 11:00,Carol,Finance
ORD-005,Ship Order,2024-01-05 17:00,Dave,Warehouse
ORD-005,Close Order,2024-01-07 09:00,Eve,Sales
ORD-006,Register Order,2024-01-08 09:00,Alice,Sales
ORD-006,Check Stock,2024-01-08 09:50,Bob,Warehouse
ORD-006,Approve Payment,2024-01-08 13:00,Carol,Finance
ORD-006,Ship Order,2024-01-09 08:00,Dave,Warehouse
ORD-006,Close Order,2024-01-10 09:00,Alice,Sales
ORD-007,Register Order,2024-01-09 09:00,Eve,Sales
ORD-007,Check Stock,2024-01-09 09:30,Bob,Warehouse
ORD-007,Backorder Item,2024-01-11 09:00,Bob,Warehouse
ORD-007,Check Stock,2024-01-14 09:00,Bob,Warehouse
ORD-007,Approve Payment,2024-01-14 11:00,Carol,Finance
ORD-007,Ship Order,2024-01-14 17:00,Dave,Warehouse
ORD-007,Close Order,2024-01-16 09:00,Eve,Sales
ORD-008,Register Order,2024-01-10 09:00,Alice,Sales
ORD-008,Check Stock,2024-01-10 09:20,Bob,Warehouse
ORD-008,Approve Payment,2024-01-10 10:00,Carol,Finance
ORD-008,Ship Order,2024-01-10 14:00,Dave,Warehouse
ORD-008,Close Order,2024-01-11 09:00,Alice,Sales
ORD-009,Register Order,2024-01-11 09:00,Eve,Sales
ORD-009,Check Stock,2024-01-11 09:30,Bob,Warehouse
ORD-009,Approve Payment,2024-01-11 10:15,Carol,Finance
ORD-009,Payment Rejected,2024-01-11 14:00,Carol,Finance
ORD-009,Approve Payment,2024-01-12 09:00,Carol,Finance
ORD-009,Ship Order,2024-01-12 15:00,Dave,Warehouse
ORD-009,Close Order,2024-01-13 09:00,Eve,Sales
ORD-010,Register Order,2024-01-12 09:00,Alice,Sales
ORD-010,Check Stock,2024-01-12 09:25,Bob,Warehouse
ORD-010,Approve Payment,2024-01-12 10:00,Carol,Finance
ORD-010,Ship Order,2024-01-12 16:00,Dave,Warehouse
ORD-010,Close Order,2024-01-13 09:00,Alice,Sales
"""

def get_sample_df():
    return pd.read_csv(io.StringIO(_SAMPLE_CSV))

print("Sample data ready.")

Sample data ready.


In [4]:
# ---- Column auto-detection (best-guess only — you confirm it below) -----
_CASE_PATTERNS = [r"case.?id", r"^case$", r"trace.?id", r"order.?id", r"ticket.?id"]
_ACT_PATTERNS = [r"activit", r"^task$", r"^event.?name$", r"^step$", r"^action$"]
_TIME_PATTERNS = [r"time.?stamp", r"^time$", r"^date$", r"datetime", r"occur"]
_RES_PATTERNS = [r"resource", r"performer", r"assignee", r"^user$", r"actor", r"employee",
                  r"^owner$", r"^role$", r"depart", r"^team$"]

def _best_match(columns, patterns):
    norm = {c: re.sub(r"[\s_\-]+", ".", c.strip().lower()) for c in columns}
    for pat in patterns:
        for c, n in norm.items():
            if re.search(pat, n):
                return c
    return columns[0] if columns else None

def guess_columns(df):
    cols = list(df.columns)
    return {
        "case": _best_match(cols, _CASE_PATTERNS),
        "activity": _best_match(cols, _ACT_PATTERNS),
        "time": _best_match(cols, _TIME_PATTERNS),
        "resource": _best_match(cols, _RES_PATTERNS),
    }

print("Column detector ready.")

Column detector ready.


In [5]:
# ---- Step 1 widgets: upload / sample button / column confirmation -------
upload_widget = widgets.FileUpload(accept=".csv,.xlsx,.xls", multiple=False,
                                    description="Upload CSV/XLSX")
sample_button = widgets.Button(description="Use sample data instead",
                                button_style="info", icon="flask")
step1_output = widgets.Output()

def _read_uploaded_file(change):
    """Handles both ipywidgets v7 and v8 FileUpload.value shapes."""
    new_val = change["new"]
    if not new_val:
        return None, None
    if isinstance(new_val, dict):  # ipywidgets v7: {filename: {...}}
        fname = list(new_val.keys())[0]
        info = new_val[fname]
        content = info["content"]
    else:  # ipywidgets v8: tuple of file-info mappings
        info = new_val[0]
        fname = info["name"]
        content = info["content"]
    content_bytes = bytes(content)
    if fname.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(io.BytesIO(content_bytes))
    else:
        df = pd.read_csv(io.BytesIO(content_bytes))
    return df, fname

def render_column_confirmation(df, source_name):
    guesses = guess_columns(df)
    cols = list(df.columns)
    none_option = "(none)"

    case_dd = widgets.Dropdown(options=cols, value=guesses["case"], description="Case ID:")
    act_dd = widgets.Dropdown(options=cols, value=guesses["activity"], description="Activity:")
    time_dd = widgets.Dropdown(options=cols, value=guesses["time"], description="Timestamp:")
    res_dd = widgets.Dropdown(options=[none_option] + cols,
                               value=guesses["resource"] if guesses["resource"] else none_option,
                               description="Resource:")
    confirm_btn = widgets.Button(description="✅ Confirm columns & prepare log",
                                  button_style="success")
    confirm_out = widgets.Output()

    def on_confirm(b):
        with confirm_out:
            clear_output()
            try:
                work = df.copy()
                work[time_dd.value] = pd.to_datetime(work[time_dd.value], errors="coerce")
                n_bad = work[time_dd.value].isna().sum()
                work = work.dropna(subset=[case_dd.value, act_dd.value, time_dd.value])
                work = work.sort_values([case_dd.value, time_dd.value])

                state["df"] = work
                state["case_col"] = case_dd.value
                state["activity_col"] = act_dd.value
                state["time_col"] = time_dd.value
                state["resource_col"] = None if res_dd.value == none_option else res_dd.value
                state["source_name"] = source_name

                n_cases = work[case_dd.value].nunique()
                n_events = len(work)
                n_acts = work[act_dd.value].nunique()
                span = f"{work[time_dd.value].min():%Y-%m-%d} to {work[time_dd.value].max():%Y-%m-%d}"

                msg = f"""**✅ Log ready: `{source_name}`**

| | |
|---|---|
| Cases | {n_cases} |
| Events | {n_events} |
| Distinct activities | {n_acts} |
| Date range | {span} |
"""
                if n_bad:
                    msg += f"\n⚠️ {n_bad} row(s) had a timestamp that couldn't be read and were dropped."
                display(Markdown(msg))
                display(Markdown("**You're set — scroll down to Step 2 and click any button to explore.**"))
            except Exception as e:
                print(f"❌ Something went wrong preparing the log: {e}")
                print("Double-check you picked the right column for each field above and try again.")

    confirm_btn.on_click(on_confirm)

    display(Markdown(f"**Loaded `{source_name}`** — {len(df)} rows, {len(df.columns)} columns.\n\n"
                      "I've guessed which column is which below — please check them (this is the one "
                      "thing worth double-checking; everything else is automatic) and click Confirm."))
    display(widgets.VBox([case_dd, act_dd, time_dd, res_dd, confirm_btn, confirm_out]))

def on_upload_change(change):
    with step1_output:
        clear_output()
        try:
            df, fname = _read_uploaded_file(change)
        except Exception as e:
            print(f"❌ Couldn't read that file: {e}")
            return
        if df is None:
            return
        render_column_confirmation(df, fname)

def on_sample_click(b):
    with step1_output:
        clear_output()
        render_column_confirmation(get_sample_df(), "sample_order_process.csv")

upload_widget.observe(on_upload_change, names="value")
sample_button.on_click(on_sample_click)

display(widgets.HBox([upload_widget, sample_button]))
display(step1_output)

Output()

---
## 🔎 Step 2 — Explore Your Process

Once Step 1 shows "Log ready", click any button below. Each one answers a
different question about the process. You can click them in any order, as
many times as you like — results replace each other in the panel below the
buttons.

In [6]:
# ---- Step 2: the exploration buttons -------------------------------------
def _log_ready():
    if state.get("df") is None:
        print("⚠️ Please load a log in Step 1 first (upload a file, or click 'Use sample data').")
        return False
    return True

def _keys():
    return dict(case_id_key=state["case_col"], activity_key=state["activity_col"],
                timestamp_key=state["time_col"])

def _render_image(save_fn, *args, out_path):
    """Saves a pm4py diagram to a PNG file and explicitly displays it.

    We deliberately avoid pm4py's view_*() functions here: they try to
    auto-detect whether they're running inside a Jupyter notebook to decide
    how to show the image, and that detection doesn't reliably work in every
    notebook environment (Google Colab in particular) — sometimes it just
    shows nothing, with no error. Saving to a file and displaying it
    ourselves works the same way everywhere.
    """
    save_fn(*args, out_path)
    display(Image(filename=out_path))

explore_output = widgets.Output()

def show_process_map(b=None):
    with explore_output:
        clear_output()
        if not _log_ready():
            return
        display(Markdown(
            "### 🗺️ Process Map (Directly-Follows Graph)\n"
            "Each box is a step; each arrow shows *'this step was directly followed by that "
            "step'*, with the number of times it happened. This is the rawest, most literal "
            "picture of how work actually moved through the process."))
        try:
            dfg, start_acts, end_acts = pm4py.discover_dfg(state["df"], **_keys())
            _render_image(pm4py.save_vis_dfg, dfg, start_acts, end_acts,
                          out_path="_pm4py_process_map.png")
        except Exception as e:
            print(f"❌ Couldn't draw the process map: {e}")

def show_bottlenecks(b=None):
    with explore_output:
        clear_output()
        if not _log_ready():
            return
        display(Markdown(
            "### ⏱️ Bottleneck Map (Performance View)\n"
            "Same map as before, but now arrows are colored/thickened by **how long** that "
            "step usually takes, not how often it happens. Darker/thicker arrows are where "
            "time is being lost — the places worth investigating first."))
        try:
            pdfg, start_acts, end_acts = pm4py.discover_performance_dfg(state["df"], **_keys())
            save_fn = getattr(pm4py, "save_vis_performance_dfg", pm4py.save_vis_dfg)
            _render_image(save_fn, pdfg, start_acts, end_acts,
                          out_path="_pm4py_bottleneck_map.png")
        except Exception as e:
            print(f"❌ Couldn't draw the bottleneck map: {e}")

def show_formal_model(b=None):
    with explore_output:
        clear_output()
        if not _log_ready():
            return
        display(Markdown(
            "### 🧩 Formal Process Model (Petri net) — optional, more technical\n"
            "This is the same process, but expressed as a formal model (boxes = steps, "
            "circles = 'states' the case can be in between steps). It's the representation "
            "used for rigorous checks like *'does this case follow an allowed order?'* — "
            "useful context, but the Process Map above is usually the more intuitive view "
            "for a first look."))
        try:
            net, im, fm = pm4py.discover_petri_net_inductive(state["df"], **_keys())
            _render_image(pm4py.save_vis_petri_net, net, im, fm,
                          out_path="_pm4py_petri_net.png")
        except Exception as e:
            print(f"❌ Couldn't draw the formal model: {e}")

def show_variants(b=None):
    with explore_output:
        clear_output()
        if not _log_ready():
            return
        display(Markdown(
            "### 🔀 Process Variants\n"
            "Not every case follows the same path. This lists the distinct *sequences of "
            "steps* seen in the log, ranked by how common they are — the top one is usually "
            "your 'happy path'; everything below it is a deviation worth understanding."))
        df, c, a = state["df"], state["case_col"], state["activity_col"]
        try:
            variants = (df.groupby(c)[a].apply(lambda x: " → ".join(x))
                          .value_counts().reset_index())
            variants.columns = ["Path", "Number of cases"]
            variants["% of cases"] = (variants["Number of cases"] / variants["Number of cases"].sum() * 100).round(1)
            top = variants.iloc[0]
            display(Markdown(f"**Most common path ('happy path'): {top['% of cases']}% of cases "
                              f"follow it, in {int(top['Number of cases'])} case(s).**"))
            display(variants.head(15))
            if len(variants) > 15:
                display(Markdown(f"...and {len(variants) - 15} more distinct path(s), "
                                  "each followed by a small number of cases."))
        except Exception as e:
            print(f"❌ Couldn't compute variants: {e}")

def show_resources(b=None):
    with explore_output:
        clear_output()
        if not _log_ready():
            return
        res_col = state.get("resource_col")
        if not res_col:
            print("ℹ️ No resource/department column was selected for this log in Step 1, "
                  "so there's nothing to break down here. Re-run Step 1 and pick one if "
                  "your data has one (e.g. who performed each step, or which team/department).")
            return
        display(Markdown(f"### 👥 Workload by `{res_col}`\n"
                          "How many steps each person/team/department handled — useful for "
                          "spotting workload imbalance or handover patterns."))
        try:
            counts = state["df"][res_col].value_counts()
            display(counts.to_frame("Number of steps handled"))
        except Exception as e:
            print(f"❌ Couldn't compute the breakdown: {e}")

def show_stats(b=None):
    with explore_output:
        clear_output()
        if not _log_ready():
            return
        df, c, a, t = state["df"], state["case_col"], state["activity_col"], state["time_col"]
        try:
            durations_hours = (df.groupby(c)[t].agg(lambda x: (x.max() - x.min()).total_seconds() / 3600))
            msg = f"""### 📊 Key Numbers

| Metric | Value |
|---|---|
| Cases | {df[c].nunique()} |
| Events | {len(df)} |
| Distinct activities | {df[a].nunique()} |
| Distinct paths (variants) | {df.groupby(c)[a].apply(tuple).nunique()} |
| Average case duration | {durations_hours.mean():.1f} hours |
| Median case duration | {durations_hours.median():.1f} hours |
| Date range | {df[t].min():%Y-%m-%d} to {df[t].max():%Y-%m-%d} |
"""
            display(Markdown(msg))
        except Exception as e:
            print(f"❌ Couldn't compute statistics: {e}")

btn_map = widgets.Button(description="🗺️ Process Map", button_style="primary")
btn_bottle = widgets.Button(description="⏱️ Bottlenecks", button_style="warning")
btn_petri = widgets.Button(description="🧩 Formal Model (advanced)")
btn_variants = widgets.Button(description="🔀 Variants / Happy Path", button_style="primary")
btn_resources = widgets.Button(description="👥 Resource Breakdown")
btn_stats = widgets.Button(description="📊 Key Numbers")
btn_clear = widgets.Button(description="🧹 Clear", button_style="danger")

btn_map.on_click(show_process_map)
btn_bottle.on_click(show_bottlenecks)
btn_petri.on_click(show_formal_model)
btn_variants.on_click(show_variants)
btn_resources.on_click(show_resources)
btn_stats.on_click(show_stats)
btn_clear.on_click(lambda b: explore_output.clear_output())

display(widgets.HBox([btn_map, btn_bottle, btn_variants, btn_resources, btn_stats, btn_petri, btn_clear]))
display(explore_output)

Output()

---
## 📖 Glossary — what am I looking at?

- **Process Map (Directly-Follows Graph / DFG)** — the literal flow of work: which step
  tends to happen right after which other step, and how often. No judgment, just facts.
- **Bottleneck Map (Performance View)** — the same map, but weighted by *time* instead of
  *frequency*. This is usually the first place to look for improvement opportunities.
- **Formal Process Model (Petri net)** — a mathematically precise version of the process
  map, used as the foundation for rigorous conformance checks (e.g. "did this case follow
  a valid order of steps?"). More technical; optional for a first pass.
- **Variant** — one distinct sequence of steps that at least one case actually followed.
  Real processes almost always have more variants than people expect going in.
- **Happy path** — the single most common variant; a useful baseline, but rarely the whole story.
- **Case duration** — how long a case took from its first recorded step to its last.

---
## 🚀 Where to go from here

This notebook is built for **live exploration in a workshop setting** — it's intentionally
simple and hands-you-the-buttons. For day-to-day use afterward (without someone opening a
notebook each time), a few natural next steps:

- A small internal web app (e.g. built with Streamlit) that does the same thing behind a
  permanent link, for people to use on their own.
- **PMTk**, the official no-code commercial product built on the same PM4Py engine, if this
  becomes an ongoing operational tool rather than a one-off exploration.
- Asking an AI assistant in plain English to run deeper PM4Py analyses (conformance
  checking against a documented SOP, root-cause analysis of delays, etc.) on an ad hoc basis.

**To try this with your own data:** scroll back to Step 1, upload your CSV, confirm the
three columns, and click through Step 2 again — everything below updates automatically.